# Research Findings & Hypothesis Testing

## Research Objective

This study investigates whether the three largest electoral regions
(based on registered voters) can predict the final national presidential
election winner.

The analysis compares six countries and evaluates:

- Prediction accuracy
- Regional agreement
- Developed vs developing country differences
- Election exceptions

This notebook presents statistical evidence and research conclusions.

In [44]:
import pandas as pd
import numpy as np

from scipy import stats

import matplotlib.pyplot as plt

In [45]:
df = pd.read_csv(
    r"D:\Election Data Project\data\processed\Elections_Feature_Engineered.csv"
)

In [46]:
df.head()

,Country,country_type,election_year,election_date,largest_electorate,largest_winner,second_electorate,second_winner,third_electorate,third_winner,...,agreement_score,prediction_confidence,top3_sweep,regional_winner_diversity,election_decade,years_since_previous_election,country_total_elections,country_type_encoded,prediction_error_category,majority_winner_region_count
0,Brazil,Developing,1989,1989-11-15,SÆo Paulo,Fernando Collor,Minas Gerais,Fernando Collor,Rio de Janeiro,Fernando Collor,...,1.000000,Very High,True,1,1980,NaN,9,0,Correct Prediction,3
1,Brazil,Developing,1994,1994-10-03,SÆo Paulo,Fernando Henrique Cardoso,Minas Gerais,Fernando Henrique Cardoso,Rio de Janeiro,Fernando Henrique Cardoso,...,1.000000,Very High,True,1,1990,5.0,9,0,Correct Prediction,3
2,Brazil,Developing,1998,1998-10-04,SÆo Paulo,Fernando Henrique Cardoso,Minas Gerais,Fernando Henrique Cardoso,Rio de Janeiro,Fernando Henrique Cardoso,...,1.000000,Very High,True,1,1990,4.0,9,0,Correct Prediction,3
3,Brazil,Developing,2002,2002-10-27,SÆo Paulo,Jos‚ Serra,Minas Gerais,Lula,Rio de Janeiro,Lula,...,0.666667,Low,False,2,2000,4.0,9,0,Correct Prediction,2
4,Brazil,Developing,2006,2006-10-29,SÆo Paulo,Geraldo Alckmin,Minas Gerais,Lula,Rio de Janeiro,Lula,...,0.666667,Low,False,2,2000,4.0,9,0,Correct Prediction,2


In [48]:
print("Total Elections:", len(df))

print(
    "Countries:",
    df["Country"].nunique()
)

Total Elections: 57
Countries: 6


## Research Questions

### Research Questions 1
How accurately can the three largest electoral regions predict the national winner?

### Research Questions 2
Do developed countries show higher prediction accuracy compared with developing countries?

### Research Questions 3
Does stronger agreement among the largest three electoral regions increase prediction success?

### Research Questions 5
Which elections represent prediction failures and why?

## SECTION 1
#### Overall Research Finding

## Overall Prediction Performance
This section measures the overall ability of the largest three electoral
regions to predict the national election outcome.

In [49]:
total_elections = len(df)

correct_predictions = df[
    "prediction_correct"
].sum()


incorrect_predictions = (
    total_elections -
    correct_predictions
)


accuracy = (
    correct_predictions /
    total_elections
) * 100


overall_result = pd.DataFrame({

    "Metric":[
        "Total Elections",
        "Correct Predictions",
        "Incorrect Predictions",
        "Accuracy (%)"
    ],

    "Value":[
        total_elections,
        correct_predictions,
        incorrect_predictions,
        round(accuracy,2)
    ]

})


overall_result

,Metric,Value
0,Total Elections,57.00
1,Correct Predictions,51.00
2,Incorrect Predictions,6.00
3,Accuracy (%),89.47


### Confidence Interval

In [50]:
confidence_interval = stats.binomtest(
    correct_predictions,
    total_elections
).proportion_ci(
    confidence_level=0.95
)


print(
    "95% Confidence Interval:"
)

print(
    confidence_interval
)

95% Confidence Interval:
ConfidenceInterval(low=0.7848363951977577, high=0.9603784641499192)


### Interpretation

The confidence interval estimates the range where the true prediction
performance is likely to exist.

A narrower interval indicates stronger certainty.

## SECTION 2
### Hypothesis 1
##### Developed Countries vs Developing Countries

## Question

Do developed countries have higher election prediction accuracy?

### Null Hypothesis (H0)

There is no difference between developed and developing countries.

### Alternative Hypothesis (H1)

Developed countries have higher prediction accuracy.

In [51]:
developed = df[
    df["country_type"]=="Developed"
]["prediction_correct"]


developing = df[
    df["country_type"]=="Developing"
]["prediction_correct"]

In [52]:
development_results = pd.DataFrame({

"Group":[
"Developed",
"Developing"
],

"Accuracy":[

developed.mean()*100,

developing.mean()*100

]

})


development_results

,Group,Accuracy
0,Developed,94.117647
1,Developing,82.608696


### Mann Whitney U Test

In [53]:
test_result = stats.mannwhitneyu(
    developed,
    developing,
    alternative="greater"
)


print(
"Statistic:",
test_result.statistic
)

print(
"P-value:",
test_result.pvalue
)

Statistic: 436.0
P-value: 0.08667673580295354


In [54]:
if test_result.pvalue < 0.05:

    print(
    "Reject H0: Evidence suggests developed countries have higher accuracy."
    )

else:

    print(
    "Fail to reject H0: Difference is not statistically significant."
    )

Fail to reject H0: Difference is not statistically significant.


### SECTION 3
#### Hypothesis 2
##### Agreement Score and Prediction Success

### Question

Does stronger agreement among the largest three electoral regions
increase prediction accuracy?

### H0

Agreement score has no relationship with prediction success.

### H1

Higher agreement increases prediction success.

In [55]:
agreement_table = (

df.groupby(
"agreement_score"
)

["prediction_correct"]

.mean()

*100

)


agreement_table

agreement_score
0.000000      0.0
0.333333      0.0
0.666667    100.0
1.000000    100.0
Name: prediction_correct, dtype: float64

In [56]:
correlation = stats.pearsonr(

df["agreement_score"],

df["prediction_correct"].astype(int)

)


print(
"Correlation:",
correlation.statistic
)


print(
"P-value:",
correlation.pvalue
)

Correlation: 0.755134485529892
P-value: 1.1487766776632883e-11


In [57]:
if correlation.pvalue < 0.05:

    print(
    "Agreement score has a statistically significant relationship with prediction success."
    )

else:

    print(
    "No statistically significant relationship detected."
    )

Agreement score has a statistically significant relationship with prediction success.


### SECTION 4
#### Country Comparison

This section identifies which countries have the strongest and weakest
predictability.

In [58]:
country_results = (

df.groupby("Country")

.agg(

Total_Elections=("Country","count"),

Correct=("prediction_correct","sum"),

Accuracy=("prediction_correct","mean"),

Average_Agreement=("agreement_score","mean")

)

)


country_results["Accuracy"] = (
country_results["Accuracy"]*100
)


country_results.sort_values(
"Accuracy",
ascending=False
)

,Total_Elections,Correct,Accuracy,Average_Agreement
Country,,,,
Portugal,10,10,100.000000,0.900000
United States,16,15,93.750000,0.770833
Brazil,9,8,88.888889,0.777778
South Korea,8,7,87.500000,0.708333
Indonesia,5,4,80.000000,0.733333
Sri Lanka,9,7,77.777778,0.740741


### SECTION 5
#### Prediction Failure Analysis

Incorrect predictions are important because they reveal situations where
large electoral regions did not represent the national outcome.

In [59]:
failures = df[
df["prediction_correct"]==False
]


failures[
[
"Country",
"election_year",
"largest_winner",
"second_winner",
"third_winner",
"national_winner",
"match_count"
]

]

,Country,election_year,largest_winner,second_winner,third_winner,national_winner,match_count
8,Brazil,2022,Jair Bolsonaro,Lula,Jair Bolsonaro,Lula,1
11,Indonesia,2014,Prabowo Subianto,Prabowo Subianto,Joko Widodo,Joko Widodo,1
31,South Korea,2022,Lee Jae-myung,Lee Jae-myung,Yoon Suk Yeol,Yoon Suk Yeol,1
33,Sri Lanka,1988,Sirimavo Bandaranaike,Sirimavo Bandaranaike,Sirimavo Bandaranaike,Ranasinghe Premadasa,0
38,Sri Lanka,2015,Mahinda Rajapaksa,Mahinda Rajapaksa,Mahinda Rajapaksa,Maithripala Sirisena,0
55,United States,2020,Joe Biden,Donald Trump,Donald Trump,Joe Biden,1


### SECTION 6
#### Research Summary Table

In [60]:
research_summary = pd.DataFrame({

    "Research Question":[
        "Overall prediction ability",
        "Developed vs developing difference",
        "Agreement impact",
        "Country variation"
    ],

    "Key Finding":[
        "89.47% prediction accuracy",
        "Developed higher (94.12% vs 82.61%), not significant",
        "Strong relationship (r=0.755, p<0.001)",
        "Portugal highest (100%), Sri Lanka lowest (77.78%)"
    ]

})

research_summary

,Research Question,Key Finding
0,Overall prediction ability,89.47% prediction accuracy
1,Developed vs developing difference,"Developed higher (94.12% vs 82.61%), not significant"
2,Agreement impact,"Strong relationship (r=0.755, p<0.001)"
3,Country variation,"Portugal highest (100%), Sri Lanka lowest (77.78%)"


## Detailed Interpretation

### Overall Prediction Ability

The three largest electorates successfully predicted the national winner in 89.47% of elections.

### Developed vs Developing Countries

Developed countries achieved higher observed accuracy (94.12%) compared with developing countries (82.61%). However, statistical testing failed to reject the null hypothesis, meaning the difference was not statistically significant.

### Agreement Impact

Agreement score showed the strongest relationship with prediction performance. The correlation coefficient was 0.755 with a p-value below 0.001, indicating a statistically significant positive relationship.

### Country Variation

Prediction performance varied across countries. Portugal achieved 100% accuracy, while Sri Lanka recorded 77.78%. These differences may be influenced by electoral structures and regional voting patterns.

In [61]:
country_results.to_csv(
r"D:\Election Data Project\data\processed\Research_Country_Results.csv"
)


research_summary.to_csv(
r"D:\Election Data Project\data\processed\Research_Summary.csv",
index=False
)


failures.to_csv(
r"D:\Election Data Project\data\processed\Prediction_Failures.csv",
index=False
)

### SECTION 7

#### Limitations

##### Dataset Size

The study contains 57 elections from six countries.
Therefore, conclusions should be interpreted carefully.

##### Electoral System Differences

Countries have different:

- Electoral systems
- District structures
- Political histories

##### Modern Electorate Selection

The largest electorates were selected using modern registered voter
patterns, which may not perfectly represent historical electorate sizes.